# Produce Tonsil Prediction CSVs

In [1]:
import sys
from pathlib import Path

start = Path.cwd().resolve()
for candidate in (start, *start.parents):
    sprint_dir = candidate / "codes" / "sprint"
    if (sprint_dir / "prediction_export.py").exists():
        PROJECT_ROOT = candidate
        if str(PROJECT_ROOT) not in sys.path:
            sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise RuntimeError(f"Cannot find codes/sprint/prediction_export.py from {start}")

# Add legacy model path for this dataset
LEGACY_DIR = PROJECT_ROOT / "codes" / "_legacy_models" / "tonsil"
if str(LEGACY_DIR) not in sys.path:
    sys.path.insert(0, str(LEGACY_DIR))

from codes.sprint.prediction_export import select_least_used_cuda_before_torch_import

select_least_used_cuda_before_torch_import()

Detected CUDA devices before torch import:
  physical=0 used=3568 MB / 11264 MB (31.7%) <-- selected as cuda:0


In [2]:
import gc
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import torch
from torch.utils.data import DataLoader

from codes.sprint.prediction_export import export_model_specs, find_project_root

# Legacy model & dataset classes
from codes._legacy_models.tonsil.model import *
from codes._legacy_models.tonsil.utils import TonsilMultimodalDataset, load_model_weights, set_seed

PROJECT_ROOT = find_project_root(Path.cwd())
print(f"Project root: {PROJECT_ROOT}")

set_seed(42)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Project root: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github
Using device: cuda:0


In [3]:
# ---- Paths ----
TRAIN_H5AD = PROJECT_ROOT / "datas" / "tonsil" / "Tonsil_1_Final.h5ad"
VAL_H5AD = PROJECT_ROOT / "datas" / "tonsil" / "Tonsil_2_Final.h5ad"
MODEL_SAVE_ROOT = PROJECT_ROOT /"datas"/ "models" / "tonsil"
OUTPUT_DIR = PROJECT_ROOT /"datas"/ "outputs" / "tonsil"

# ---- Model specs ----
MODEL_SPECS = [
    {"label": "HE_Only", "model_class": Model_SchemeA2_HE_Only,
     "model_dir_candidates": ["HE_Only"],
     "output_prefix": "tonsil_HE_Only"},
    {"label": "A2", "model_class": Model_SchemeA2,
     "model_dir_candidates": ["A2"],
     "output_prefix": "tonsil_A2"},
     {"label": "RNA_Only", "model_class": Model_RNA_Only,
     "model_dir_candidates": ["RNA_Only"],
     "output_prefix": "tonsil_RNA_Only"},
]
BATCH_SIZE = 32

for required_path in [TRAIN_H5AD, VAL_H5AD, MODEL_SAVE_ROOT]:
    if not required_path.exists():
        raise FileNotFoundError(required_path)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [4]:
ad_train = sc.read_h5ad(TRAIN_H5AD, backed="r")
ad_val = sc.read_h5ad(VAL_H5AD, backed="r")
p_train = [str(x) for x in list(ad_train.uns["protein_names"])] if "protein_names" in ad_train.uns else [str(i) for i in range(ad_train.obsm["protein_expression_log"].shape[1])]
p_val = [str(x) for x in list(ad_val.uns["protein_names"])] if "protein_names" in ad_val.uns else [str(i) for i in range(ad_val.obsm["protein_expression_log"].shape[1])]
common_proteins = sorted(set(p_train).intersection(p_val))
del ad_train, ad_val
gc.collect()

val_dataset = TonsilMultimodalDataset(str(VAL_H5AD), target_proteins=common_proteins)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

NUM_GENES = val_dataset.rna_data.shape[1]
NUM_TARGETS = val_dataset.protein_data.shape[1]
target_names = [str(x) for x in list(val_dataset.protein_names)]

print(f"Inference config: {NUM_GENES} genes, {NUM_TARGETS} proteins, {len(val_dataset)} validation spots")
print(target_names)

Loading data from: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/tonsil/Tonsil_2_Final.h5ad ...
 Filtering proteins: intersection of 29 and target 29...
 Filtered to 29 proteins.
Inference config: 18085 genes, 29 proteins, 4908 validation spots
['ACTA2-1', 'BCL2-1', 'CCR7-1', 'CD14-1', 'CD163-1', 'CD19-1', 'CD27-1', 'CD274-1', 'CD3E-1', 'CD4-1', 'CD40-1', 'CD68-1', 'CD8A-1', 'CEACAM8-1', 'CR2-1', 'CXCR5-1', 'EPCAM-1', 'FCGR3A-1', 'HLA-DRA', 'ITGAM-1', 'ITGAX-1', 'KRT5-1', 'MS4A1-1', 'PAX5-1', 'PCNA-1', 'PDCD1-1', 'PECAM1-1', 'SDC1-1', 'VIM-1']


In [5]:
saved_df, pred_df, target_df = export_model_specs(
    MODEL_SPECS,
    MODEL_SAVE_ROOT,
    OUTPUT_DIR,
    val_loader,
    device,
    NUM_TARGETS,
    NUM_GENES,
    target_names,
    load_model_weights,
)

display(saved_df)
# display(pred_df.head())
# display(target_df.head())

[HE_Only] saved predictions: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/tonsil/tonsil_HE_Only_predictions.csv
[HE_Only] saved targets:     /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/tonsil/tonsil_HE_Only_targets.csv
[A2] saved predictions: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/tonsil/tonsil_A2_predictions.csv
[A2] saved targets:     /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/tonsil/tonsil_A2_targets.csv
[RNA_Only] saved predictions: /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/tonsil/tonsil_RNA_Only_predictions.csv
[RNA_Only] saved targets:     /media/yuzhi/My PSSD/spatialProtein/spatialProtein/Github/datas/outputs/tonsil/tonsil_RNA_Only_targets.csv


,Model,Weights,ModelDir,Predictions,Targets,Rows,TargetsCount
0,HE_Only,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,HE_Only,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,4908,29
1,A2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,A2,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,4908,29
2,RNA_Only,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,RNA_Only,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,/media/yuzhi/My PSSD/spatialProtein/spatialPro...,4908,29
